In [1]:

#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

import time

from pandas import ExcelWriter

import re

import pdfplumber

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options



In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'BG FSC'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__))

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



Running BG FSC Web Scraping Tool v.1.1


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------



#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

		 "selectedDestinationId": tempfolder,

    	 "version": 2}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

#driver = webdriver.Chrome(executable_path="..\\chromedriver.exe",options=chromeOptions)

driver.maximize_window()

In [4]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

def scrollinAndClick(xpath,key_press=False):
    if len(xpath) != 0 :

        for times in range(60):

            try:
                driver.find_element(By.XPATH, xpath).click()

                sleep(5)

                break
            except:
                # print(f"[ERROR] : trying {times+1}/10 to key press 'DOWN' (scrolling)")
                sleep(5)
                if key_press:                    

                    driver.find_element(By.TAG_NAME, 'body').send_keys(key_press)
        else:   
            raise Exception(f'[ERROR] : Failed scrollin Or Click on xpath element : {xpath}')

        

def find_zip_code(string):
    match = re.search(r'\d{4}', string)
    if match:
        return match.group()
    else:
        return None
    
def find_phone_number(string):
    match = re.search(r'\+359\d+', string)
    if match:
        return match.group()
    else:
        return None


def scroll_to_bottom(driver):
    # Get scroll height
    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:
        # Scroll down to the bottom
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # Wait to load the page
        time.sleep(2)

        # Calculate new scroll height and compare with last scroll height
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------


regdict={'BG FSC 1': 'https://www.fsc.bg/en/investment-avtivity/lists-of-supervised-entities/management-companies/', 
         'BG FSC 2': 'https://www.fsc.bg/en/investment-avtivity/lists-of-supervised-entities/alternative-investment-fund-managers/' ,
         'BG FSC 3': 'https://www.fsc.bg/en/investment-avtivity/lists-of-supervised-entities/investment-firms/',
        'BG FSC 4': 'https://www.fsc.bg/en/social-insurance-activity/supervised-entities/pension-insurance-companies/'
        }


Typology={
        'BG FSC 1': 'List of Management Companies', 
        'BG FSC 2': 'List of Alternative Investment Fund Managers' ,
        'BG FSC 3': 'List of Investment Firms',
        'BG FSC 4': 'List of Pension Insurance Companies',
         }
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')
		  

In [ ]:
# %%
#------------------------------------------------ Begin_Main ----------------------------------------

for index, reg in enumerate(regdict):
    inner_links = []
    driver.delete_all_cookies()
    inner_links = []
    print('Working with {}.'.format(reg))
    driver.get(regdict[reg])
    sleep(1)
    scroll_to_bottom(driver)
    sleep(2)
    soup=BeautifulSoup(driver.page_source, "html.parser")
    sleep(1)
    table = soup.find('table',id='myTable')
    sleep(2)
    tbody=table.find('tbody')
    trs = tbody.find_all('tr')
    if reg == 'BG FSC 1':
        for index, tr in enumerate(trs[1:]):
            name = tr.find_all('td')[0]
            try:
                contact_info = tr.find_all('td')[1]
            except:
                continue
            name = name.text
            #print(name)
            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')
            
            #print(contact_info)
            data = contact_info.get_text(separator='\n').strip()
            extracted_data = data.split('\n')
            
            for iindex, info in enumerate(extracted_data):
                #print(info)
                if iindex == 0:
                    #print('---Áddress---', info)
                    if 'phone' in info:
                        check_index = info.index('phone')
                        address = info[:check_index]
                    else:
                        address = info
                    sqldict['Address_1'].append(address)
                    zip = find_zip_code(address)
                    sqldict['Zip'].append(zip)
                    Cntry = address.split(',')[0].strip()
                    if Cntry.upper() == 'BULGARIA':
                        sqldict['Cntry'].append('BG')
                if 'phone' in info:
                    index_phone = info.index('phone')
                    #print('---Phone---', info[7:])
                    sqldict['Phone'].append(info[index_phone+7:])
                if 'fax' in info:
                    index_fax = info.index('fax')
                    #print('-----Fax----', info[4:])
                    sqldict['Fax'].append(info[index_fax+4:])
            emails = ''
            websites= ''
            hyper_links = contact_info.find_all('a')
            for hyper_link in hyper_links:
                if 'http'  in hyper_link.text:
                    # websites = ''.join(hyper_link.text)
                    websites += hyper_link.text+' '
                elif 'www' in hyper_link.text:
                    #websites = ''.join(hyper_link.text)
                    websites += hyper_link.text+' '
                else:
                    emails += hyper_link.text+' '
            sqldict['Website'].append(websites.strip())        
            sqldict['Email'].append(emails.strip())
            sqldict = bourange_same_length_array(sqldict)
            
    elif reg == 'BG FSC 2' :            
        for index, tr in enumerate(trs[1:]):
            name = tr.find_all('td')[0]
            try:
                contact_info = tr.find_all('td')[1]
            except:
                continue
            name = name.text
            #print(name)
            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')
            
            #print(contact_info)
            data = contact_info.get_text(separator='\n').strip()
            extracted_data = data.split('\n')
            
            for iindex, info in enumerate(extracted_data):
                #print(info)
                if iindex == 0:
                    #print('---Áddress---', info)
                    tel_index = info.find('+ 359')
                    if 'phone' in info:
                        check_index = info.index('phone')
                        address = info[:check_index]
                        #print(address)
                    elif 'Phone' in info:
                        check_index = info.index('Phone')
                        address = info[:check_index]
                        #print(address)                       
                    elif tel_index!=-1:
                        address = info[:tel_index]
                        #print(info[:tel_index])
                    else:
                        address = info
                        #print(address)
                    sqldict['Address_1'].append(address)
                    
                    zip = find_zip_code(address)
                    sqldict['Zip'].append(zip)
                    #Cntry = address.split(',')[0].strip()
                    if 'BULGARIA' or 'Bulgaria' in info:
                        sqldict['Cntry'].append('BG')
                    
                    
                if 'phone'  in info:
                    index_phone = info.index('phone')
                    #print('---Phone---', info[7:])
                    sqldict['Phone'].append(info[index_phone+7:])
                elif 'Phone' in info:
                    index_phone = info.index('Phone')
                    #print('---Phone---', info[7:])
                    sqldict['Phone'].append(info[index_phone+7:])
               
                if 'fax' in info:
                    index_fax = info.index('fax')
                    #print('-----Fax----', info[4:])
                    sqldict['Fax'].append(info[index_fax+4:])
            emails = ''
            websites= ''
            hyper_links = contact_info.find_all('a')
            for hyper_link in hyper_links:
                if 'http'  in hyper_link.text:
                    # websites = ''.join(hyper_link.text)
                    websites += hyper_link.text+' '
                elif 'www' in hyper_link.text:
                    #websites = ''.join(hyper_link.text)
                    websites += hyper_link.text+' '
                else:
                    emails += hyper_link.text+' '
            sqldict['Website'].append(websites.strip())        
            sqldict['Email'].append(emails.strip())
            sqldict = bourange_same_length_array(sqldict)       
            
    elif reg == 'BG FSC 3':
        for index, tr in enumerate(trs[1:]):
            name = tr.find_all('td')[0]
            try:
                contact_info = tr.find_all('td')[1]
            except:
                continue
            name = name.text
            #print(name)
            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')
            
            #print(contact_info)
            data = contact_info.get_text(separator='\n').strip()
            extracted_data = data.split('\n')
            
            for iindex, info in enumerate(extracted_data):
            
                if 'tel' in info or 'Tel' in info:
                    if info.find('tel') != -1:
                        tel_info = info[info.find('tel'):]
                        extra_info1 = tel_info.find('fax')
                        extra_info2 = tel_info.find('E-mail')
                        extra_info3 = tel_info.find('e-mail')
                        if extra_info1!=-1:
                            #print(info[info.find('tel'):extra_info1])
                            sqldict['Phone'].append(info[info.find('tel')+4:extra_info1])
                            
                        elif extra_info2!=-1:
                            sqldict['Phone'].append(info[info.find('tel')+4:extra_info2])
                        elif extra_info3!=-1:
                            sqldict['Phone'].append(info[info.find('tel')+4:extra_info3])
                        else:
                            sqldict['Phone'].append(info[info.find('tel')+4:])
                    elif info.find('Tel'):
                        if info.find('Tel') != -1:
                            tel_info = info[info.find('Tel'):]
                            extra_info1 = tel_info.find('fax')
                            extra_info2 = tel_info.find('E-mail')
                            extra_info3 = tel_info.find('e-mail')
                            extra_info4 = tel_info.find('Fax')
                        if extra_info1!=-1:
                            sqldict['Phone'].append(info[info.find('Tel')+4:extra_info1])
                        elif extra_info2!=-1:
                            sqldict['Phone'].append(info[info.find('Tel')+4:extra_info2])
                        elif extra_info3!=-1:
                            sqldict['Phone'].append(info[info.find('Tel')+4:extra_info3])
                        elif extra_info4!=-1:
                            sqldict['Phone'].append(info[info.find('Tel')+4:extra_info4])
                        else:
                            sqldict['Phone'].append(info[info.find('Tel')+4:])
                            
                websites= ''
                email = ''
                hyper_links = contact_info.find_all('a')
                for hyper_link in hyper_links:
                    if 'http'  in hyper_link.text:
                        # websites = ''.join(hyper_link.text)
                        websites += hyper_link.text+' '
                    elif 'www' in hyper_link.text:
                        #websites = ''.join(hyper_link.text)
                        websites += hyper_link.text+' '

                    if '@'  in hyper_link.text:
                        # websites = ''.join(hyper_link.text)
                        email += hyper_link.text+' '

                    else:
                        email += hyper_link.text+' '
            address=extracted_data[0] 
            sqldict['Address_1'].append(address)
            zip = find_zip_code(address) 
            sqldict['Zip'].append(zip)
            if 'BULGARIA' or 'Bulgaria' in address:
                sqldict['Cntry'].append('BG')  
            #print(websites,email)    
            sqldict['Website'].append(websites.strip())        
            sqldict['Email'].append(email.strip())        
            sqldict = bourange_same_length_array(sqldict)
    elif reg == 'BG FSC 4': 
        for index, tr in enumerate(trs[1:]):
            name = tr.find_all('td')[0]
            try:
                contact_info = tr.find_all('td')[1]
            except:
                continue
            name = name.text
            #print(name)
            sqldict['Name'].append(name)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')
            
            #print(contact_info)
            data = contact_info.get_text(separator='\n').strip()
            extracted_data = data.split('\n')
            

            websites= ''
            email = ''
            hyper_links = contact_info.find_all('a')
            for hyper_link in hyper_links:
                if 'http'  in hyper_link.text:
                    # websites = ''.join(hyper_link.text)
                    websites += hyper_link.text+' '
                elif 'www' in hyper_link.text:
                    #websites = ''.join(hyper_link.text)
                    websites += hyper_link.text+' '

                if '@'  in hyper_link.text:
                    # websites = ''.join(hyper_link.text)
                    email += hyper_link.text+' '

                else:
                    email += hyper_link.text+' '
            address = extracted_data[0]
            print(websites,email)  
            for iindex, info in enumerate(extracted_data):
                 if 'Phone' in info :
                     
                    if info.find('Phone') != -1:
                        tel_info = info[info.find('Phone'):]
                        extra_info1 = tel_info.find('Fax')

                        if extra_info1!=-1:
                            #print(info[info.find('tel'):extra_info1])
                            sqldict['Phone'].append(info[info.find('Phone')+6:extra_info1])
                            sqldict['Fax'].append(info[extra_info1:])
                        else:
                            sqldict['Phone'].append(info[info.find('Phone')+6:])
            address=extracted_data[0] 
            sqldict['Address_1'].append(address)
            zip = find_zip_code(address) 
            sqldict['Zip'].append(zip)
            if 'BULGARIA' or 'Bulgaria' in address:
                sqldict['Cntry'].append('BG')  
            #print(websites,email)    
            sqldict['Website'].append(websites.strip())        
            sqldict['Email'].append(email.strip())        
            sqldict = bourange_same_length_array(sqldict)                     
                
            
            
            
                   

Working with BG FSC 1.
ALARIC CAPITAL
IMPACT CAPITAL JSC.
ACTIVA ASSET MANAGEMENT
ASTRA ASSET MANAGEMENT
VARCHEV MANAGING COMPANY
SMART FUND ASSET MANAGEMENT
DV ASSET MANAGEMENT
COMPASS INVEST JSC
DSK ASSETS MANAGEMENT
EXPAT ASSET MANAGEMENT
KBC INVESTMENT MANAGEMENT
CAPMAN ASSET MANAGEMENT
ZLATEN LEV CAPITAL
INVEST CAPITAL
ELANA FOND MANAGEMENT
KAROLL CAPITAL MANAGEMENT
MUNICIPAL BANK ASSET MANAGEMENT
FFBH ASSET MANAGEMENT
REAL FINANCE ASSET MANAGEMENT
SELECT ASET MANAGEMENT
SKY ASSET MANAGEMENT
STRATEGY ASET MANAGEMENT
CONCORD ASSET MANAGEMENT
TEXIM ASSET MANAGEMENT
EF ASSET MANAGEMENT
CCB ASSET MANAGEMENT
YG MARKET FOND MANAGEMENT
INVEST FOND MANAGEMENT
KBC ASSET MANAGEMENT – KLON
THRACIAN INVEST INC.
BLUESMART INVESTMENTS AD
Working with BG FSC 2.
BORG FINANCE
2A Ivam Stracimirov Str, office 500, 9000 Varna, Bulgaria 
VITOSHA VENTURE PARTNERS
18 “132” Str, Floor 6, Apt. 11, 1113 Sofia 
VF ALTERNATIVE
193 G.S.Rakovski Str, Entrance A, Floor 1, Apt. 2, 1142 Sofia, Bulgaria 
IMPETUS C

In [7]:

# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 124 values.
Key 'priority' has 124 values.
Key 'ListLabel' has 124 values.
Key 'Typology' has 124 values.
Key 'EntryType' has 124 values.
Key 'Name' has 124 values.
Key 'InternalID_1' has 124 values.
Key 'InternalID_1_type' has 124 values.
Key 'InternalID_2' has 124 values.
Key 'InternalID_2_type' has 124 values.
Key 'InternalID_3' has 124 values.
Key 'InternalID_3_type' has 124 values.
Key 'CoType' has 124 values.
Key 'License_Type' has 124 values.
Key 'Address_1' has 124 values.
Key 'Address_2' has 124 values.
Key 'City' has 124 values.
Key 'Zip' has 124 values.
Key 'Cntry' has 124 values.
Key 'Phone' has 124 values.
Key 'Fax' has 124 values.
Key 'Website' has 124 values.
Key 'Email' has 124 values.
Key 'RegulationType' has 124 values.
Key 'RegulationTypeCode' has 124 values.
Key 'RegulationDate' has 124 values.
Key 'CancellationDate' has 124 values.
Key 'RegCtry' has 124 values.
Key 'RegCode' has 124 values.
Key 'ListCode' has 124 values.
Key 'ListLanguage' has 124 v

In [8]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\5\ipykernel_19196\3068940208.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [9]:
df.to_csv('total_2025_v1.csv')

In [26]:
for iindex, info in enumerate(extracted_data):
    print(info.split('e-mail'))



['69 Bulgaria Blvd., fl.12, office 12 – Sofia 1404 – Bulgaria']
['е-mail: ']
['sofia@activtrades.eu']


In [9]:
extracted_data[0]

'1172 Sofia, Dianabad Quarter, 1 G. M. Dimitrov Blvd.'